# 🏪 Vending-Bench 2 — Setup Verification

This notebook verifies that the **VB2 environment** is correctly installed and the
server is reachable. We'll run a mini episode: reset the simulation, inspect our
starting balance, adjust a price, request a supplier quote, and advance a few days
to observe sales & revenue.

## What is Vending-Bench 2?

VB2 is a **simulated vending-machine business** built on the [OpenEnv](https://github.com/openenv) framework.
An agent manages a single vending machine over 365 simulated days:

| Aspect | Details |
|---|---|
| **Products** | soda, water, candy, chips, sandwich |
| **Starting capital** | $500 |
| **Goal** | Maximize ending cash balance |
| **Actions** | set prices, order stock, restock machine, negotiate with suppliers, manage customer complaints |
| **Challenges** | demand varies by weather/season/day, suppliers can be unreliable, daily fees erode margins |

The environment exposes an **MCP tool-calling interface** — the agent picks a tool
and arguments each step, exactly like an LLM function-calling loop.

## 1 — Start the Server

In a **separate terminal**, start the VB2 server:

```bash
cd /Users/robertamanfu/dev/hackathon/cerebral_valley/openenv
source .venv/bin/activate
PYTHONPATH=vendsim_vb2 python -m openenv.server vendsim_vb2.mcp_env:VB2MCPEnvironment --port 8000
```

Wait until you see `Uvicorn running on http://0.0.0.0:8000` before running the
cells below.

## 2 — Connect & Reset

In [ ]:
import sys, os

# Ensure vendsim_vb2 is importable
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "vendsim_vb2"))

from vendsim_vb2 import VB2Client

In [ ]:
SERVER_URL = "http://localhost:8000"

env = VB2Client(base_url=SERVER_URL)
env.connect()
print("✅ Connected to VB2 server")

In [ ]:
reset_result = env.reset()
print("✅ Environment reset")
print(f"   Done:   {reset_result.done}")
print(f"   Reward: {reset_result.reward}")

## 3 — Explore: List Tools & Check Balance

In [ ]:
tools = env.list_tools()
print(f"📦 {len(tools)} tools available:\n")
for t in tools:
    print(f"  • {t.name:30s} {t.description}")

In [ ]:
balance = env.check_balance()
print(f"💰 Starting balance: {balance}")

In [ ]:
inventory = env.check_storage_inventory()
print(f"📋 Storage inventory: {inventory}")

## 4 — Act: Set a Price & Request a Quote

In [ ]:
price_result = env.set_price("soda", 1.75)
print(f"🏷️  Price update: {price_result}")

In [ ]:
quote = env.request_supplier_quote("chips", 20)
print(f"📧 Supplier quote request: {quote}")

In [ ]:
env.write_scratchpad("Day 1: Set soda price to $1.75, requested chips quote.")
print("📝 Scratchpad updated")

## 5 — Simulate: Advance a Few Days

In [ ]:
NUM_DAYS = 5

print(f"⏩ Advancing {NUM_DAYS} days...\n")
for i in range(NUM_DAYS):
    day_result = env.wait_for_next_day(output_tokens=0)
    print(f"  Day {i+1}: {day_result}")

print()

## 6 — Results

In [ ]:
final_balance = env.check_balance()
print(f"💰 Final balance after {NUM_DAYS} days: {final_balance}")

notes = env.read_scratchpad()
print(f"📝 Scratchpad: {notes}")

In [ ]:
env.close()
print("🔌 Disconnected from VB2 server")
print("\n✅ Setup verification complete!")

---

## Next Steps

- **`01_baseline_agent.ipynb`** — Build a simple heuristic agent
- **`02_llm_agent.ipynb`** — Wire up an LLM with tool-calling
- **`03_training_loop.ipynb`** — RL-style training with reward signal